# Biohub × Linajea: random node hide pilot

Attach the Biohub competition data, enable a GPU and Internet, then run all cells. This pilot hides about 20% of known GEFF nodes, trains for 2,000 iterations using visible annotations only, reconstructs a 20-frame 3D ROI, and saves every result under `/kaggle/working/biohub_random_hide`. Unknown predictions are **not** treated as false positives because the original GT is sparse.

In [ ]:
from pathlib import Path
import json, subprocess, sys
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_random_hide')
REPO = Path('/kaggle/working/linajea')
SAMPLE = '6bba_05b6850b'
HIDE_RATE = 0.20
SEED = 42
ITERATIONS = 2000
BRANCH = 'biohub-io'
PINNED_COMMIT = None  # Set to the SHA after pushing for a fixed rerun
if not (DATA_DIR / f'{SAMPLE}.zarr').is_dir() or not (DATA_DIR / f'{SAMPLE}.geff').is_dir():
    raise FileNotFoundError(f'Missing Zarr/GEFF pair for {SAMPLE} in {DATA_DIR}')
ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', DATA_DIR, 'Sample:', SAMPLE, 'Output:', ROOT)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU for this training run')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
COMMIT = PINNED_COMMIT or subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', COMMIT], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(ROOT / 'commit.txt').write_text(COMMIT + '\n')
print('Pinned code commit:', COMMIT, 'GPU:', torch.cuda.get_device_name(0))
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)
pip('-r', REPO / 'requirements-biohub-io.txt')
pip('-r', REPO / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO, '--no-deps')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide', 'split',
                '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--output-dir', str(ROOT), '--hide-rate', str(HIDE_RATE),
                '--seed', str(SEED)], cwd=REPO, check=True)
split = json.loads((ROOT / 'split.json').read_text())
print('Visible nodes:', len(split['visible_node_ids']),
      'Hidden nodes:', len(split['hidden_node_ids']),
      'Visible edges:', len(split['visible_edges']),
      'Hidden edges:', len(split['hidden_edges']))
print('Pilot ROI start / shape:', split['roi_start'], split['roi_shape'])

In [ ]:
TRAIN_DIR = ROOT / 'train'
TRAIN_DIR.mkdir(exist_ok=True)
command = [sys.executable, '-m', 'run_scripts.train_biohub_smoke',
           '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
           '--tracksfile', str(ROOT / 'visible.geff'),
           '--output-dir', str(TRAIN_DIR),
           '--iterations', str(ITERATIONS), '--checkpoint-stride', '500']
with (TRAIN_DIR / 'train.log').open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        if 'Batch: iteration=' in line or 'checkpoint' in line.lower() or 'Traceback' in line:
            print(line.rstrip())
    if proc.wait() != 0:
        raise RuntimeError(f'Training failed; inspect {TRAIN_DIR / "train.log"}')
checkpoint = TRAIN_DIR / f'train_net_checkpoint_{ITERATIONS}'
print('Checkpoint saved:', checkpoint, checkpoint.stat().st_size, 'bytes')

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'reconstruct', '--data-dir', str(DATA_DIR),
                '--sample', SAMPLE, '--output-dir', str(ROOT),
                '--iterations', str(ITERATIONS)], cwd=REPO, check=True)
print('Prediction files:', sorted(p.name for p in (ROOT / 'prediction').iterdir()))

In [ ]:
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'evaluate', '--data-dir', str(DATA_DIR),
                '--sample', SAMPLE, '--output-dir', str(ROOT)],
               cwd=REPO, check=True)
metrics = json.loads((ROOT / 'evaluation' / 'metrics.json').read_text())
print('Reminder: hidden-confirmed rate is a lower bound, not true precision.')

In [ ]:
from IPython.display import display, Image, HTML, FileLink
subprocess.run([sys.executable, '-m', 'run_scripts.biohub_random_hide',
                'plot', '--data-dir', str(DATA_DIR),
                '--sample', SAMPLE, '--output-dir', str(ROOT)],
               cwd=REPO, check=True)
picture = ROOT / 'plots' / 'comparison.png'
display(Image(filename=str(picture)))
print('Saved comparison:', picture)
print('All output files:', [str(p.relative_to(ROOT)) for p in ROOT.rglob('*') if p.is_file()])
html = ROOT / 'plots' / 'comparison_3d.html'
if html.exists():
    display(FileLink(str(html)))